# Проверка экспорта

Этот notebook проверяет сохранение итогового рейтинга в Excel-файл.

In [1]:
from importlib import import_module
from pathlib import Path
import sys
import pandas as pd

# Определяем корень проекта.
project_root = Path.cwd()
if not (project_root / "data" / "trade.xlsx").exists():
    project_root = project_root.parent

# Добавляем корень проекта в пути импорта.
sys.path.insert(0, str(project_root))

loader = import_module("src.1_data_loader.loader")
preprocessing = import_module("src.2_preprocessing.preprocessing")
indicators = import_module("src.3_indicators.indicators")
normalization = import_module("src.4_normalization.normalization")
model = import_module("src.5_model.model")
exporter = import_module("src.7_export.exporter")

In [2]:
# Выполняем полный расчет рейтинга.
calculation_year = 2025
clipping_mode = "1-99"

raw_data = loader.load_excel_data(project_root / "data" / "trade.xlsx")
prepared_data = preprocessing.preprocess_trade_data(raw_data)
yearly_trade = preprocessing.make_yearly_trade_table(prepared_data)
country_import = preprocessing.make_country_import_table(prepared_data)
indicator_values = indicators.calculate_indicators(yearly_trade, country_import, calculation_year)
normalized_values = normalization.normalize_indicators(indicator_values, clipping_mode)
ranking, ahp_info = model.calculate_priority_ranking(normalized_values)

In [3]:
# Сохраняем результат в Excel.
output_path = project_root / "outputs" / f"ranking_{calculation_year}_test.xlsx"

exporter.export_ranking_to_excel(
    ranking,
    ahp_info,
    output_path,
    calculation_year,
    clipping_mode,
)

output_path

WindowsPath('c:/Users/admin/OneDrive/Рабочий стол/import-substitution-priority/outputs/ranking_2025_test.xlsx')

In [4]:
# Проверяем листы и содержимое файла.
excel_file = pd.ExcelFile(output_path)
expected_sheets = {"Ranking", "AHP Weights", "Parameters"}

assert output_path.exists()
assert set(excel_file.sheet_names) == expected_sheets

ranking_from_excel = pd.read_excel(output_path, sheet_name="Ranking")
weights_from_excel = pd.read_excel(output_path, sheet_name="AHP Weights")
parameters_from_excel = pd.read_excel(output_path, sheet_name="Parameters")

assert len(ranking_from_excel) == len(ranking)
assert set(weights_from_excel["Criterion"]) == {"C1", "C2", "C3", "C4"}
assert "consistency_ratio" in set(parameters_from_excel["Parameter"])

print("Excel export is correct")

Excel export is correct
